# GKG Feature Construction

Purpose: Apply the theme whitelist derived from EDA to a larger sample (100K companies, 
12-month window), and construct company-level features.

Pipeline: Load sample, BigQuery retrieval, AC mapping, Novel prefix check, Apply whitelist, Company-level features, Sanity checks


`jupyter nbconvert --to html "01 EDA + Data PreProcessing/GKG_theme_feature_Construction.ipynb"`

In [1]:
import os
import re
import ahocorasick
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from datetime import datetime, timedelta
from google.cloud import bigquery

In [2]:
# BigQuery authentication
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../key/lloyds-gdelt-500117-a700e8484a10.json"
client = bigquery.Client(project="lloyds-gdelt-500117")

In [40]:
# Paths
DATA_PATH = "../01 EDA + Data PreProcessing/01_CompaniesSelected/UKcompanies_active_account_category_sample_100k.csv"
OUTPUT_DIR = Path("../output/gkg_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GKG_RAW_CSV = OUTPUT_DIR / "gkg_raw_100k_12m.csv"
GKG_RAW_PARQUET = OUTPUT_DIR / "gkg_raw_100k_12m.parquet"
MAPPED_PARQUET = OUTPUT_DIR / "mapped_100k_12m.parquet"
FEATURE_TABLE_PQ = OUTPUT_DIR / "gkg_features_100k.parquet"
FEATURE_TABLE_CSV= OUTPUT_DIR / "gkg_features_100k.csv"
FEATURE_TABLE_THEME_SCORE_PQ = OUTPUT_DIR / "gkg_features_theme_score_100k.parquet"
FEATURE_TABLE_THEME_SCORE_CSV= OUTPUT_DIR / "gkg_features_theme_score_100k.csv"

In [4]:
# GKG query window: 12 months
GKG_END = datetime.strptime("2026-05-01", "%Y-%m-%d")
GKG_START = GKG_END - timedelta(days=365)
GKG_START_STR = GKG_START.strftime("%Y-%m-%d")
GKG_END_STR = GKG_END.strftime("%Y-%m-%d")
print(f"GKG window: {GKG_START_STR} - {GKG_END_STR}")

GKG window: 2025-05-01 - 2026-05-01


## 1. Load 100K Company Sample

Load the 100K company table and generate cleaned search names.

In [10]:
sample_100k = pd.read_csv(DATA_PATH, dtype=str)
print(f"Sample size: {len(sample_100k)}")
print(f"\nSector distribution:")
print(sample_100k['primary_sector'].value_counts())

Sample size: 100000

Sector distribution:
primary_sector
Technology, legal & professional        26432
Real Estate                             24727
Wholesale & Retail                      17841
Fast growth & emerging sector            9761
Healthcare                               7913
Manufacturing                            7066
Public sector, education & charities     4697
Agriculture                              1563
Name: count, dtype: int64


In [11]:
# Company name cleaning (identical to gdelt EDA notebook)
SUFFIX_PATTERN = re.compile(
    r'\b(LIMITED|LTD|PLC|LLP|L\.L\.P|L\.L\.C|LLC|UNLIMITED)\b\.?',
    flags=re.IGNORECASE
)

def clean_company_name(name: str) -> str:
    """Strip legal suffixes and normalise to a search-ready form."""
    original_normalized = re.sub(r'[^\w\s]', ' ', name)
    original_normalized = re.sub(r'\s+', ' ', original_normalized).strip().lower()
    
    name = SUFFIX_PATTERN.sub('', name)
    name = re.sub(r'&', ' ', name)
    name = re.sub(r'[^\w\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip().lower()

    if not name:
        return original_normalized

    return name

sample_100k['search_name'] = sample_100k['CompanyName'].apply(clean_company_name)

# Deduplicate for the query pattern
main_names = sample_100k['search_name'].dropna().unique().tolist()
print(f"Total companies: {len(sample_100k)}")
print(f"Unique search names: {len(main_names)}")

Total companies: 100000
Unique search names: 99995


Cleaned company names in the 100K sample yield 99995 unique search names (5 duplicates), so cleaning-induced name collisions are negligible.

## 2. BigQuery Data Retrieval

Same batch query strategy as the EDA notebook: combine all company names into one regex, 
run a single SQL query. Results are streamed in chunks and appended to a single CSV.

In [12]:
# Build regex pattern
def build_company_pattern(names: list):
    """Combine company names into a single regex with word boundaries."""
    valid_names = [n.lower().strip() for n in names if n and n.strip()]
    escaped = [re.escape(n) for n in valid_names]
    pattern = r'\b(' + '|'.join(escaped) + r')\b'
    return pattern, valid_names

In [16]:
# Fields retained: DATE, DocumentIdentifier, V2Themes, V2Organizations, V2Tone
# V2Counts and Amounts are not used in features.

def query_gkg_batch_dryrun(company_names: list,
                           start_date: str = GKG_START_STR,
                           end_date: str = GKG_END_STR,
                           uk_only: bool = False) -> float:
    """Dry run — returns estimated scan size in GB without executing."""
    pattern, valid_names = build_company_pattern(company_names)
    if len(valid_names) < len(company_names):
        print(f"Input: {len(company_names)}, In pattern: {len(valid_names)}")

    uk_filter = "AND SourceCommonName LIKE '%.uk'" if uk_only else ""
    sql = f"""
    SELECT
      DATE,
      DocumentIdentifier,
      V2Themes,
      V2Organizations,
      V2Tone
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONTIME BETWEEN TIMESTAMP('{start_date}') AND TIMESTAMP('{end_date}')
      AND REGEXP_CONTAINS(LOWER(V2Organizations), @pattern)
      {uk_filter}
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("pattern", "STRING", pattern)],
        dry_run=True,
        use_query_cache=False,
    )
    job = client.query(sql, job_config=job_config)
    gb = job.total_bytes_processed / 1e9
    print(f"Estimated scan: {gb:.2f} GB")
    return gb


def query_gkg_batch(company_names: list,
                    start_date: str = GKG_START_STR,
                    end_date: str = GKG_END_STR,
                    uk_only: bool = False,
                    output_path: str = str(GKG_RAW_CSV)):
    """
    Execute batch query, streaming results as chunks appended to CSV.
    Run query_gkg_batch_dryrun first to confirm scan size.
    """
    # Clean up existing file
    if Path(output_path).exists():
        Path(output_path).unlink()
        print(f"Removed existing file: {output_path}")

    pattern, _ = build_company_pattern(company_names)
    uk_filter = "AND SourceCommonName LIKE '%.uk'" if uk_only else ""
    sql = f"""
    SELECT
      DATE,
      DocumentIdentifier,
      V2Themes,
      V2Organizations,
      V2Tone
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONTIME BETWEEN TIMESTAMP('{start_date}') AND TIMESTAMP('{end_date}')
      AND REGEXP_CONTAINS(LOWER(V2Organizations), @pattern)
      {uk_filter}
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("pattern", "STRING", pattern)]
    )
    query_job = client.query(sql, job_config=job_config)
    rows = query_job.result()

    is_first_chunk = True
    total_rows = 0
    chunk_num = 0
    for chunk in rows.to_dataframe_iterable():
        chunk.to_csv(output_path, mode='a', index=False, header=is_first_chunk)
        is_first_chunk = False
        total_rows += len(chunk)
        chunk_num += 1
        if chunk_num % 10 == 0:
            print(f"Chunk {chunk_num}, total rows: {total_rows}")
    print(f"Finished. Total chunks: {chunk_num}, total rows: {total_rows}")

### 2.1 Dry Run

Confirm the scan size before executing the actual query. Free tier is 1TB/month, and this query should be well within budget.


In [17]:
query_gkg_batch_dryrun(main_names)

Estimated scan: 148.04 GB


148.041024678

In [ ]:
# Find the company whose cleaned name became empty
empty_names = sample_100k[
    sample_100k['search_name'].isna() | 
    (sample_100k['search_name'].str.strip() == '')
]
print(f"Empty search_name count: {len(empty_names)}")
print(empty_names[['CompanyNumber', 'CompanyName', 'search_name']])

Empty search_name count: 1
      CompanyNumber  CompanyName search_name
88966      15881414  PLC LIMITED            


### 2.2 Execute Query

Run the actual query.

In [20]:
# query_gkg_batch(main_names)

# Failed, pattern too large.

In [19]:
# Use a 1-day window to test regex compilation with minimal scan
TEST_START = "2026-06-19"
TEST_END = "2026-06-20"  # test with only 1 day

for batch_size in [50000, 34000, 30000, 20000, 15000]:
    subset = main_names[:batch_size]
    try:
        query_gkg_batch(subset, 
                        start_date=TEST_START, 
                        end_date=TEST_END,
                        output_path=str(OUTPUT_DIR / f"pattern_test_{batch_size}.csv"))
        print(f"Batch size {batch_size} passes regex compilation")
        break
    except Exception as e:
        if "pattern too large" in str(e):
            print(f"Batch size {batch_size} fails: pattern too large")
        else:
            print(f"Batch size {batch_size} fails: {type(e).__name__}: {e}")

Batch size 50000 fails: pattern too large
Finished. Total chunks: 1, total rows: 3223
Batch size 34000 passes regex compilation


In [22]:
def query_gkg_batch_split(company_names: list, batch_size: int = 34000,
                           output_dir: Path = OUTPUT_DIR):
    """
    Split company names into batches and query each separately,
    to work around BigQuery's regex pattern size limit.
    """
    total_batches = (len(company_names) + batch_size - 1) // batch_size
    print(f"Splitting {len(company_names)} names into {total_batches} batches of {batch_size}")

    for i in range(total_batches):
        start = i * batch_size
        end = start + batch_size
        batch = company_names[start:end]
        batch_output = output_dir / f"gkg_raw_100k_12m_batch{i:02d}.csv"
        
        if batch_output.exists():
            print(f"[Batch {i+1}/{total_batches}] Already exists, skipping: {batch_output}")
            continue
        
        print(f"\n[Batch {i+1}/{total_batches}] Querying {len(batch)} names → {batch_output}")
        query_gkg_batch(batch, output_path=str(batch_output))
    
    print(f"\nAll batches done.")

In [23]:
query_gkg_batch_split(main_names)

Splitting 99995 names into 3 batches of 34000

[Batch 1/3] Querying 34000 names → ..\output\gkg_features\gkg_raw_100k_12m_batch00.csv
Chunk 10, total rows: 40809
Chunk 20, total rows: 82598
Chunk 30, total rows: 123277
Chunk 40, total rows: 165434
Chunk 50, total rows: 207834
Chunk 60, total rows: 248915
Chunk 70, total rows: 291088
Chunk 80, total rows: 332165
Chunk 90, total rows: 373977
Chunk 100, total rows: 416480
Chunk 110, total rows: 457433
Chunk 120, total rows: 499601
Chunk 130, total rows: 540726
Chunk 140, total rows: 583081
Chunk 150, total rows: 625351
Chunk 160, total rows: 666267
Chunk 170, total rows: 707884
Chunk 180, total rows: 749224
Chunk 190, total rows: 791235
Finished. Total chunks: 192, total rows: 800003

[Batch 2/3] Querying 34000 names → ..\output\gkg_features\gkg_raw_100k_12m_batch01.csv
Chunk 10, total rows: 43564
Chunk 20, total rows: 87307
Chunk 30, total rows: 131082
Chunk 40, total rows: 174632
Chunk 50, total rows: 218760
Chunk 60, total rows: 262313

In [24]:
# Load the batch files and combine them into a single CSV, deduplicating by DocumentIdentifier
batch_files = sorted(OUTPUT_DIR.glob("gkg_raw_100k_12m_batch*.csv"))
print(f"Found {len(batch_files)} batch files")

dfs = []
for f in batch_files:
    df = pd.read_csv(f)
    print(f"  {f.name}: {len(df)} rows")
    dfs.append(df)

# Concatenate and deduplicate by DocumentIdentifier
combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal before deduplicate: {len(combined)}")

combined = combined.drop_duplicates(subset='DocumentIdentifier')
print(f"Total after deduplicate: {len(combined)}")

# Save
combined.to_csv(GKG_RAW_CSV, index=False)
print(f"Saved to: {GKG_RAW_CSV}")

# Also save as parquet
combined.to_parquet(GKG_RAW_PARQUET, compression='snappy')
print(f"Saved to: {GKG_RAW_PARQUET}")

Found 3 batch files
  gkg_raw_100k_12m_batch00.csv: 800003 rows
  gkg_raw_100k_12m_batch01.csv: 1832799 rows
  gkg_raw_100k_12m_batch02.csv: 1896385 rows

Total before deduplicate: 4529187
Total after deduplicate: 4345189
Saved to: ..\output\gkg_features\gkg_raw_100k_12m.csv
Saved to: ..\output\gkg_features\gkg_raw_100k_12m.parquet


### 2.3 Load and Convert to Parquet

Load the CSV and save as parquet for faster downstream loading.
The CSV is kept as a raw backup.

In [6]:
main_gkg = pd.read_csv(GKG_RAW_CSV)
print(f"Loaded {len(main_gkg)} rows")
print(f"Columns: {main_gkg.columns.tolist()}")
main_gkg.head()

Loaded 4345189 rows
Columns: ['DATE', 'DocumentIdentifier', 'V2Themes', 'V2Organizations', 'V2Tone']


,DATE,DocumentIdentifier,V2Themes,V2Organizations,V2Tone
0,20251010234500,https://www.bozemandailychronicle.com/ap_news/...,"AFFECT,1221;AFFECT,2019;CRISISLEX_T02_INJURED,...","National Women Law Center,1783;Human Services,...","-7.92079207920792,2.14521452145215,10.06600660..."
1,20250521233000,https://www.foxbusiness.com/lifestyle/burger-k...,"TAX_FNCACT_KING,11;TAX_FNCACT_KING,299;TAX_FNC...","Burger King,11;Burger King,299;Burger King,557...","3.7037037037037,3.7037037037037,0,3.7037037037..."
2,20260123233000,https://www.romesentinel.com/ap/national/us-co...,"CRISISLEX_C03_WELLBEING_HEALTH,5413;TAX_FNCACT...","Georgetown University,800;United States,4294;R...","-3.35497835497836,3.03030303030303,6.385281385..."
3,20251106234500,https://www.globenewswire.com/news-release/202...,"USPEC_POLICY1,328;USPEC_POLICY1,1886;USPEC_POL...","Pravo Ventures Inc,2416","5.69105691056911,7.31707317073171,1.6260162601..."
4,20260112234500,https://biz.heraldcorp.com/article/10654046,"EDUCATION,1437;SOC_POINTSOFINTEREST_UNIVERSITY...","Korea Creative Content Agency,1032;Ministry Of...","3.28947368421053,4.27631578947368,0.9868421052..."


## 3. Article-to-Company Mapping

Use Aho-Corasick automaton to identify which companies each article mentions. Same as GDELT EDA notebook.

In [8]:
def map_articles_to_companies_fast(gkg_df: pd.DataFrame,
                                    company_names: list) -> pd.DataFrame:
    """Map articles to companies via Aho-Corasick automaton."""
    A = ahocorasick.Automaton()
    valid_count = 0
    # Deduplicate company names before adding
    unique_names = set()
    for name in company_names:
        if name and isinstance(name, str):
            cleaned = name.lower().strip()
            if cleaned:
                unique_names.add(cleaned)

    for name in unique_names:
        A.add_word(name, name)
        valid_count += 1

    if valid_count == 0:
        return pd.DataFrame()

    print(f"Building automaton for {valid_count} unique names.")
    A.make_automaton()
    print("Automaton built.")

    cols = ['V2Organizations', 'DATE', 'DocumentIdentifier',
            'V2Themes', 'V2Tone']
    arrays = {c: gkg_df[c].values for c in cols}
    n = len(gkg_df)

    rows = []
    for i in range(n):
        orgs_raw = arrays['V2Organizations'][i]
        if not isinstance(orgs_raw, str):
            continue

        orgs_lower = orgs_raw.lower()
        length = len(orgs_lower)
        matched = set()

        for end_index, name in A.iter(orgs_lower):
            start_index = end_index - len(name) + 1
            left_ok = (start_index == 0) or not orgs_lower[start_index - 1].isalnum()
            right_ok = (end_index == length - 1) or not orgs_lower[end_index + 1].isalnum()
            if left_ok and right_ok:
                matched.add(name)

        for comp in matched:
            rows.append({
                'company_search_name': comp,
                'DATE': arrays['DATE'][i],
                'DocumentIdentifier': arrays['DocumentIdentifier'][i],
                'V2Themes': arrays['V2Themes'][i],
                'V2Organizations': orgs_raw,
                'V2Tone': arrays['V2Tone'][i],
            })

        if (i + 1) % 100000 == 0:
            print(f"  Scanned {i+1}/{n} articles, {len(rows)} matches so far")

    return pd.DataFrame(rows)

In [12]:
main_mapped = map_articles_to_companies_fast(main_gkg, main_names)
print(f"\nInput articles: {len(main_gkg)}")
print(f"Mapped rows: {len(main_mapped)}")
print(f"Distinct matched companies: {main_mapped['company_search_name'].nunique()}")

Building automaton for 99995 unique names.
Automaton built.
  Scanned 100000/4345189 articles, 113370 matches so far
  Scanned 200000/4345189 articles, 226985 matches so far
  Scanned 300000/4345189 articles, 340180 matches so far
  Scanned 400000/4345189 articles, 453518 matches so far
  Scanned 500000/4345189 articles, 566534 matches so far
  Scanned 600000/4345189 articles, 679964 matches so far
  Scanned 700000/4345189 articles, 793353 matches so far
  Scanned 800000/4345189 articles, 906581 matches so far
  Scanned 900000/4345189 articles, 1013794 matches so far
  Scanned 1000000/4345189 articles, 1121009 matches so far
  Scanned 1100000/4345189 articles, 1228220 matches so far
  Scanned 1200000/4345189 articles, 1335376 matches so far
  Scanned 1300000/4345189 articles, 1442608 matches so far
  Scanned 1400000/4345189 articles, 1549824 matches so far
  Scanned 1500000/4345189 articles, 1656926 matches so far
  Scanned 1600000/4345189 articles, 1764093 matches so far
  Scanned 170

In [13]:
# Save mapped result as parquet
main_mapped.to_parquet(MAPPED_PARQUET, compression='snappy')
print(f"Saved: {MAPPED_PARQUET}")

Saved: ..\output\gkg_features\mapped_100k_12m.parquet


## 4. Prefix Check

The 100K sample with 12-month window may contain theme prefixes not observed in the original EDA (10K sample, 3-month window). Check whether any new prefixes appear and whether they need manual classification.

In [14]:
# Compute article-level frequency and prefix set for the new data
article_frequency = Counter()
for themes_str in main_mapped['V2Themes'].dropna():
    article_themes = set()
    for theme in str(themes_str).split(';'):
        theme_name = theme.split(',')[0].strip()
        if theme_name:
            article_themes.add(theme_name)
    for theme_name in article_themes:
        article_frequency[theme_name] += 1

new_prefixes = Counter()
for theme, count in article_frequency.items():
    prefix = theme.split('_')[0] if '_' in theme else theme
    new_prefixes[prefix] += count

print(f"Distinct themes: {len(article_frequency)}")
print(f"Distinct prefixes: {len(new_prefixes)}")

Distinct themes: 21369
Distinct prefixes: 174


In [16]:
# Old prefix set from EDA (168 prefixes observed on 10K sample)
OLD_PREFIXES = {'ACT',
 'AFFECT',
 'AGRICULTURE',
 'AID',
 'ALLIANCE',
 'APPOINTMENT',
 'ARMEDCONFLICT',
 'ARREST',
 'ASSASSINATION',
 'AUSTERITY',
 'AVIATION',
 'BAN',
 'BLACK',
 'BLOCKADE',
 'BORDER',
 'BULLYING',
 'CEASEFIRE',
 'CHARASMATIC',
 'CHECKPOINT',
 'CLAIM',
 'CLOSURE',
 'CONFISCATION',
 'CONSTITUTIONAL',
 'CORRUPTION',
 'CRIME',
 'CRISISLEX',
 'CRM',
 'CURFEW',
 'CYBER',
 'DEATH',
 'DEFECTION',
 'DELAY',
 'DEMOCRACY',
 'DISABILITY',
 'DISASTER',
 'DISCRIMINATION',
 'DISPLACED',
 'DRONES',
 'DRUG',
 'ECON',
 'EDUCATION',
 'ELECTION',
 'EMERG',
 'ENV',
 'EPU',
 'ETH',
 'EVACUATION',
 'EXHUMATION',
 'EXILE',
 'EXTREMISM',
 'FIREARM',
 'FOOD',
 'FREESPEECH',
 'FUELPRICES',
 'GEN',
 'GENDER',
 'GENERAL',
 'GENTRIFICATION',
 'GOV',
 'GRIEVANCES',
 'HARASSMENT',
 'HATE',
 'HEALTH',
 'HUMAN',
 'IDEOLOGY',
 'IMMIGRATION',
 'IMPEACHMENT',
 'INCOME',
 'INEQUALITY',
 'INFO',
 'INFRASTRUCTURE',
 'INSURGENCY',
 'INTERNET',
 'JIHAD',
 'JUSTICE',
 'KIDNAP',
 'KILL',
 'LANDMINE',
 'LEADER',
 'LEG',
 'LEGALIZE',
 'LEGISLATION',
 'LGBT',
 'LITERACY',
 'MANMADE',
 'MARITIME',
 'MED',
 'MEDIA',
 'MEDICAL',
 'MIL',
 'MILITARY',
 'MOVEMENT',
 'NATURAL',
 'NEGOTIATIONS',
 'NEW',
 'ORGANIZED',
 'PEACEKEEPING',
 'PERSECUTION',
 'PIRACY',
 'POLITICAL',
 'POPULATION',
 'POVERTY',
 'POWER',
 'PRIVATIZATION',
 'PROPAGANDA',
 'PROPERTY',
 'PROTEST',
 'PUBLIC',
 'RAIL',
 'RAPE',
 'RATIFY',
 'REBELLION',
 'REBELS',
 'RECRUITMENT',
 'REFUGEES',
 'REL',
 'RELATIONS',
 'RELEASE',
 'RELIGION',
 'RESIGNATION',
 'RETALIATE',
 'RETIREMENT',
 'RETIREMENTS',
 'ROAD',
 'RURAL',
 'SANCTIONS',
 'SANITATION',
 'SCANDAL',
 'SCIENCE',
 'SECURITY',
 'SEIGE',
 'SEIZE',
 'SELF',
 'SHORTAGE',
 'SICKENED',
 'SLFID',
 'SLUMS',
 'SMUGGLING',
 'SOC',
 'SOVEREIGNTY',
 'STATE',
 'STRIKE',
 'SUICIDE',
 'SURVEILLANCE',
 'TAKE',
 'TAX',
 'TECH',
 'TERROR',
 'TORTURE',
 'TOURISM',
 'TRAFFIC',
 'TRANSPARENCY',
 'TREASON',
 'TRIAL',
 'UNEMPLOYMENT',
 'UNGP',
 'UNREST',
 'UNSAFE',
 'URBAN',
 'USPEC',
 'VANDALIZE',
 'VETO',
 'VIOLENT',
 'WATER',
 'WB',
 'WHISTLEBLOWER',
 'WMD',
 'WOUND'}

# Identify new prefixes
observed_prefixes = set(new_prefixes.keys())
novel = observed_prefixes - OLD_PREFIXES
missing = OLD_PREFIXES - observed_prefixes

print(f"Observed prefixes in new data: {len(observed_prefixes)}")
print(f"New prefixes (not in EDA): {len(novel)}")
print(f"EDA prefixes missing in new data: {len(missing)}")

if novel:
    print(f"New prefixes with article frequency:")
    for p in sorted(novel, key=lambda x: -new_prefixes[x]):
        print(f"  {p}: {new_prefixes[p]}")

Observed prefixes in new data: 174
New prefixes (not in EDA): 6
EDA prefixes missing in new data: 0
New prefixes with article frequency:
  SEPARATISTS: 6858
  PIPELINE: 592
  UNGOVERNED: 174
  LOCUSTS: 158
  PHONE: 142
  POL: 110


### Handling New Prefixes

**New prefixes are all low-frequency and non-business**: They fall outside the whitelist by default.


## 5. Apply Theme Whitelist

Use the whitelist derived from EDA.

In [18]:
# From EDA notebook
RELATED_PREFIXES = {
    'ECON',
    'RECRUITMENT',
    'UNEMPLOYMENT',
    'SHORTAGE',
    'FUELPRICES',
    'AUSTERITY',
    'PRIVATIZATION',
    'SANCTIONS',
    'INCOME',
    'STRIKE',
    'CLOSURE',
}

WB_KEEP = {
    'WB_2670_JOBS',
    'WB_2689_JOBS_DIAGNOSTICS',
    'WB_2769_JOBS_STRATEGIES',
    'WB_1921_PRIVATE_SECTOR_DEVELOPMENT',
    'WB_405_BUSINESS_CLIMATE',
    'WB_2530_BUSINESS_ENVIRONMENT',
    'WB_346_COMPETITIVE_INDUSTRIES',
    'WB_818_INDUSTRY_POLICY_AND_REAL_SECTORS',
    'WB_698_TRADE',
    'WB_1920_FINANCIAL_SECTOR_DEVELOPMENT',
}

In [19]:
# Build final business-relevant theme set
BUSINESS_RELATED_THEMES = set()
for theme in article_frequency:
    prefix = theme.split('_')[0] if '_' in theme else theme
    if prefix in RELATED_PREFIXES:
        BUSINESS_RELATED_THEMES.add(theme)
BUSINESS_RELATED_THEMES.update(WB_KEEP)
#BUSINESS_RELATED_THEMES.update(TAX_KEEP)

print(f"Total business-relevant themes: {len(BUSINESS_RELATED_THEMES)}")

Total business-relevant themes: 367


In [20]:
# Coverage check at article level
articles_with_relevant = 0
articles_with_themes_but_no_relevant = 0
articles_no_themes = 0

for themes_str in main_mapped['V2Themes'].fillna(''):
    if not themes_str:
        articles_no_themes += 1
        continue
    themes_in_article = {t.split(',')[0].strip()
                         for t in str(themes_str).split(';')
                         if t.strip()}
    if not themes_in_article:
        articles_no_themes += 1
    elif themes_in_article & BUSINESS_RELATED_THEMES:
        articles_with_relevant += 1
    else:
        articles_with_themes_but_no_relevant += 1

total = len(main_mapped)
print(f"Articles with relevant themes: {articles_with_relevant} ({articles_with_relevant/total:.1%})")
print(f"Articles with themes but no match: {articles_with_themes_but_no_relevant} ({articles_with_themes_but_no_relevant/total:.1%})")
print(f"Articles with no themes: {articles_no_themes} ({articles_no_themes/total:.1%})")

Articles with relevant themes: 2201357 (47.9%)
Articles with themes but no match: 2077965 (45.2%)
Articles with no themes: 320322 (7.0%)


## 6. Company-Level Feature Construction

Build three features per company. Unmatched companies get zero/False values.


Features:
- `gkg_features_available` (bool): whether the company has any GKG match
- `has_business_news` (bool): whether any matched article contains a business-relevant theme
- `business_article_count` (int): number of matched articles containing business themes 

In [21]:
# Step 1: compute per-company aggregates from mapped table
def article_has_business_theme(themes_str, whitelist):
    """Check whether an article's V2Themes contains any whitelist theme."""
    if not isinstance(themes_str, str) or not themes_str:
        return False
    themes_in_article = {t.split(',')[0].strip()
                         for t in themes_str.split(';')
                         if t.strip()}
    return bool(themes_in_article & whitelist)

# Mark each mapped row with whether the article is business-relevant
main_mapped['is_business'] = main_mapped['V2Themes'].apply(
    lambda s: article_has_business_theme(s, BUSINESS_RELATED_THEMES)
)

# Aggregate per company
company_aggregates = main_mapped.groupby('company_search_name').agg(
    total_article_count=('DocumentIdentifier', 'count'),
    business_article_count=('is_business', 'sum'),
).reset_index()

company_aggregates['has_business_news'] = company_aggregates['business_article_count'] > 0
company_aggregates['business_article_count'] = company_aggregates['business_article_count'].astype(int)

print(f"Companies with at least one match: {len(company_aggregates)}")
print(company_aggregates.head())

Companies with at least one match: 5710
  company_search_name  total_article_count  business_article_count  \
0              100318                    4                       0   
1                1201                78573                   34676   
2                1336                72223                   33596   
3               14930                  590                     321   
4                1933                54043                   27305   

   has_business_news  
0              False  
1               True  
2               True  
3               True  
4               True  


In [22]:
# Step 2: left join back to full 100K sample table
feature_table = sample_100k[[
    'CompanyNumber', 'CompanyName', 'search_name',
    'primary_sector', 'Accounts_AccountCategory',
]].copy()

feature_table = feature_table.merge(
    company_aggregates[['company_search_name', 'business_article_count', 'has_business_news']],
    left_on='search_name',
    right_on='company_search_name',
    how='left'
).drop(columns=['company_search_name'])

# Fill missing values for unmatched companies
feature_table['gkg_features_available'] = feature_table['business_article_count'].notna()
feature_table['business_article_count'] = feature_table['business_article_count'].fillna(0).astype(int)
feature_table['has_business_news'] = feature_table['has_business_news'].fillna(False)

print(f"Feature table shape: {feature_table.shape}")
print(f"\nHead rows:")
feature_table.head()

Feature table shape: (100000, 8)

Head rows:


C:\Users\86132\AppData\Local\Temp\ipykernel_21628\1557603772.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feature_table['has_business_news'] = feature_table['has_business_news'].fillna(False)


,CompanyNumber,CompanyName,search_name,primary_sector,Accounts_AccountCategory,business_article_count,has_business_news,gkg_features_available
0,13209628,TICKETY BOO TEETH LIMITED,tickety boo teeth,Healthcare,UNAUDITED ABRIDGED,0,False,False
1,13887674,TECH INFUSION LIMITED,tech infusion,Fast growth & emerging sector,TOTAL EXEMPTION FULL,0,False,False
2,13408899,THE SCC ACADEMY LIMITED,the scc academy,Fast growth & emerging sector,TOTAL EXEMPTION FULL,0,False,False
3,SC374368,GRACEFRUIT LIMITED,gracefruit,Wholesale & Retail,TOTAL EXEMPTION FULL,0,False,False
4,13675979,LINHAM LIMITED,linham,Wholesale & Retail,TOTAL EXEMPTION FULL,0,False,False


In [23]:
# Save as both parquet (for analysis) and CSV (for portability)
feature_table.to_parquet(FEATURE_TABLE_PQ, compression='snappy')
feature_table.to_csv(FEATURE_TABLE_CSV, index=False)
print(f"Saved parquet: {FEATURE_TABLE_PQ}")
print(f"Saved CSV: {FEATURE_TABLE_CSV}")

Saved parquet: ..\output\gkg_features\gkg_features_100k.parquet
Saved CSV: ..\output\gkg_features\gkg_features_100k.csv


### 6.1 Name Ambiguity Flag

The `business_article_count` distribution reveals extreme values (max ~620k, 99th percentile ~4,600). Inspection of top-ranked companies shows the driver: cleaned search names collide with common English words or well-known non-target entities.

Examples
- SUPREME PLC (real UK company) - search_name 'supreme' , also matches US streetwear brand "Supreme"
- AMERICAN LTD - search_name 'american', matches almost any US-related article
- 1961 LIMITED - search_name '1961' , matches year references
- Common surname companies, matches unrelated people

In [5]:
feature_table = pd.read_parquet(FEATURE_TABLE_PQ)

In [20]:
covered = feature_table[feature_table['has_business_news']]
print(f"Companies with business news: {len(covered)} ({len(covered)/len(feature_table):.2%})")
print(f"\nbusiness_article_count distribution:")
print(covered['business_article_count'].describe(
    percentiles=[.25, .5, .75, .9, .95, .99]
))

Companies with business news: 4223 (4.22%)

business_article_count distribution:
count      4223.000000
mean        521.291262
std       11687.222631
min           1.000000
25%           1.000000
50%           3.000000
75%          14.000000
90%          86.800000
95%         344.900000
99%        4611.920000
max      622519.000000
Name: business_article_count, dtype: float64


In [7]:
top_by_count = feature_table[feature_table['has_business_news']].sort_values(
    'business_article_count', ascending=False
).head(30)
print(top_by_count[['CompanyName', 'search_name', 'business_article_count']])

                    CompanyName        search_name  business_article_count
90173               SUPREME PLC            supreme                  622519
45081              AMERICAN LTD           american                  392028
21103        HUMAN SERVICES LTD     human services                   98505
22728        ATLANTIC + LIMITED           atlantic                   50784
36255              EMIRATES LTD           emirates                   47792
98238          DEUTSCHE LIMITED           deutsche                   46903
42367             WHILE LIMITED              while                   46638
37337               516 LIMITED                516                   45650
9338                  LIGHT LTD              light                   44420
68614               757 LIMITED                757                   41446
67539           WELFARE LIMITED            welfare                   38627
52220               962 LIMITED                962                   38302
23050              1201 L

We do not attempt strict entity disambiguation. Instead we flag "likely ambiguous" search names using two combined signals:

- **Name structure**: very short single-token names, or single tokens that are common English words, tend to collide with irrelevant entities.

- **Observed match volume**: a normal UK company rarely generates hundreds of business-relevant news articles in 12 months. Extreme match counts strongly suggest collision with a more famous same-named entity.

Neither signal alone is decisive, but together they give a workable filter. This is a pragmatic flag, not a rigorous classification. It will produce both false positives (genuine active companies incorrectly flagged) and false negatives (true collisions missed).


In [18]:
def is_likely_ambiguous(row):
    """
    Flag likely name-ambiguity cases based on name structure and observed match volume.
    
    Rules (any triggers = flagged):
    1. Empty or missing search_name
    2. search_name shorter than 6 characters
    3. Single-token search_name shorter than 8 characters
    4. business_article_count > 300 in a 12-month window
    
    """
    name = row['search_name']
    count = row['business_article_count']
    
    # Rule 1: empty name
    if not isinstance(name, str) or not name.strip():
        return True
    
    name = name.strip()
    
    # Rule 2: very short overall
    if len(name) < 6:
        return True
    
    # Rule 3: single token that is short
    tokens = name.split()
    if len(tokens) == 1 and len(name) < 8:
        return True
    
    # Rule 4: extreme match volume
    if count > 300:
        return True
    
    return False


feature_table['likely_ambiguous'] = feature_table.apply(is_likely_ambiguous, axis=1)

n_flagged = feature_table['likely_ambiguous'].sum()
print(f"Companies flagged as likely_ambiguous: {n_flagged} ({n_flagged/len(feature_table):%})")

Companies flagged as likely_ambiguous: 6981 (6.981000%)


In [ ]:
# Break down the flag by trigger rule to understand what's driving it
def flag_reason(row):
    name = row['search_name']
    count = row['business_article_count']
    reasons = []
    if not isinstance(name, str) or not name.strip():
        reasons.append('empty_name')
    else:
        n = name.strip()
        if len(n) < 6:
            reasons.append('too_short')
        tokens = n.split()
        if len(tokens) == 1 and len(n) < 8:
            reasons.append('single_short_token')
    if count > 200:
        reasons.append('extreme_count')
    return ','.join(reasons) if reasons else 'not_flagged'

flag_breakdown = feature_table.apply(flag_reason, axis=1).value_counts()
print("Flag reason breakdown:")
print(flag_breakdown)

Flag reason breakdown:
not_flagged                                   92989
single_short_token                             4070
too_short,single_short_token                   2445
too_short                                       203
extreme_count                                   151
too_short,single_short_token,extreme_count       75
single_short_token,extreme_count                 65
too_short,extreme_count                           2
Name: count, dtype: int64


In [ ]:
# Sanity check
flagged_covered = feature_table[
    feature_table['likely_ambiguous'] & feature_table['has_business_news']
]
unflagged_covered = feature_table[
    ~feature_table['likely_ambiguous'] & feature_table['has_business_news']
]

print(f"Flagged companies with business news: {len(flagged_covered)}")
print(f"  business_article_count stats:")
print(f"    median: {flagged_covered['business_article_count'].median():.0f}")
print(f"    max:    {flagged_covered['business_article_count'].max():.0f}")
print()
print(f"Unflagged companies with business news: {len(unflagged_covered)}")
print(f"  business_article_count stats:")
print(f"    median: {unflagged_covered['business_article_count'].median():.0f}")
print(f"    max:    {unflagged_covered['business_article_count'].max():.0f}")

Flagged companies with business news: 1126
  business_article_count stats:
    median: 8
    max:    622519

Unflagged companies with business news: 3097
  business_article_count stats:
    median: 2
    max:    300


In [ ]:
# check the top 20 unflagged companies by business_article_count
unflagged_top = feature_table[
    (~feature_table['likely_ambiguous']) & 
    (feature_table['has_business_news'])
].sort_values('business_article_count', ascending=False).head(20)

print("Top 20 UNFLAGGED companies by business_article_count:")
print(unflagged_top[['CompanyName', 'search_name', 'business_article_count']].to_string())

Top 20 UNFLAGGED companies by business_article_count:
                       CompanyName                 search_name  business_article_count
73141        GOLD RESERVES LIMITED               gold reserves                     300
41174            NETWORK CHINA LTD               network china                     294
90954            RELEVANCE LIMITED                   relevance                     293
40743  MENTAL HEALTH FIRST LIMITED         mental health first                     292
74474                STORY ART LTD                   story art                     291
77639        FUTURE ENERGY LIMITED               future energy                     291
38413           ASSA ABLOY LIMITED                  assa abloy                     290
48624            SANDERSON LIMITED                   sanderson                     288
28471       INVESCO GLOBAL LIMITED              invesco global                     285
30976                 CHARISMA LTD                    charisma              

In [13]:
top_by_count = feature_table[feature_table['has_business_news']].sort_values(
    'business_article_count', ascending=False
).head(30)
print(top_by_count[['CompanyName', 'business_article_count', 'likely_ambiguous']])

                    CompanyName  business_article_count  likely_ambiguous
90173               SUPREME PLC                  622519              True
45081              AMERICAN LTD                  392028              True
21103        HUMAN SERVICES LTD                   98505              True
22728        ATLANTIC + LIMITED                   50784              True
36255              EMIRATES LTD                   47792              True
98238          DEUTSCHE LIMITED                   46903              True
42367             WHILE LIMITED                   46638              True
37337               516 LIMITED                   45650              True
9338                  LIGHT LTD                   44420              True
68614               757 LIMITED                   41446              True
67539           WELFARE LIMITED                   38627              True
52220               962 LIMITED                   38302              True
23050              1201 LIMITED       

The name ambiguity flag captures 6.98% of companies (6981 out of 100k). Structural rules (short or single-word names) account for 93% of flagged cases, with the remaining 7% caught by the article count threshold alone. Among flagged companies with business news, the median article count is 8; among unflagged, it is 2. The maximum unflagged article count is exactly 300 (the threshold), while the maximum flagged count reaches 622,519 (**superme**). The filter effectively separates typical UK companies from likely name-collision cases without requiring rigorous entity disambiguation.

In [ ]:
# Save as both parquet (for analysis) and CSV (for portability)
feature_table.to_parquet(FEATURE_TABLE_PQ, compression='snappy')
feature_table.to_csv(FEATURE_TABLE_CSV, index=False)
print(f"Saved parquet: {FEATURE_TABLE_PQ}")
print(f"Saved CSV: {FEATURE_TABLE_CSV}")

### 6.2 Compute V2theme score

**Score Strategy:**

For each company we combine three GKG features into a single score (0-100) using a piecewise rule:

- **No GKG match (94.29% of the sample)**: NaN. This is a data-availability signal — most companies simply don't appear in news, and downstream scoring should not penalise them.

- **Has GKG match but no business-relevant themes (1.49%)**: also NaN. No positive signal to score.

- **Has business news (4.22%)**: score = 20 + 80 * tanh(count / 30), ranging from 20 (base) to 100.

- **Has business news but flagged as `likely_ambiguous`**: score capped at 25. Suspicious matches receive a low-confidence signal, not a top ranking.

In [31]:
covered = feature_table[feature_table['has_business_news']].copy()
for sf in [10, 20, 50, 100]:
    covered[f'score_sf{sf}'] = 20 + 80 * np.tanh(covered['business_article_count'] / sf)
    
print(covered[[f'score_sf{sf}' for sf in [10, 20, 50, 100]]].describe(
    percentiles=[.25, .5, .75, .9, .95, .99]
))

        score_sf10   score_sf20   score_sf50  score_sf100
count  4223.000000  4223.000000  4223.000000  4223.000000
mean     56.837332    47.832886    38.689402    33.569170
std      28.945736    28.604475    25.953008    23.162845
min      27.973440    23.996670    21.599787    20.799973
25%      27.973440    23.996670    21.599787    20.799973
50%      43.305009    31.910803    24.794248    22.399280
75%      90.828132    68.349422    41.832406    31.127396
90%      99.999995    99.972790    95.180313    76.028067
95%     100.000000   100.000000    99.999837    99.838592
99%     100.000000   100.000000   100.000000   100.000000
max     100.000000   100.000000   100.000000   100.000000


In [34]:
unflagged_covered = feature_table[
    (~feature_table['likely_ambiguous']) & 
    (feature_table['has_business_news'])
].copy()
print(f"\nbusiness_article_count distribution(unflagged):")
print(unflagged_covered['business_article_count'].describe(
    percentiles=[.1, .25, .5, .75, .9, .95, .99]
))


business_article_count distribution(unflagged):
count    3097.000000
mean       13.981595
std        35.156314
min         1.000000
10%         1.000000
25%         1.000000
50%         2.000000
75%         9.000000
90%        34.000000
95%        69.000000
99%       193.120000
max       300.000000
Name: business_article_count, dtype: float64


In [33]:
unflagged_covered['raw_score'] = 20 + 80 * np.tanh(
    unflagged_covered['business_article_count'] / 50
)
print("Score distribution for unflagged companies:")
print(unflagged_covered['raw_score'].describe(
    percentiles=[.1, .25, .5, .75, .9, .95]
))

Score distribution for unflagged companies:
count    3097.000000
mean       33.578924
std        20.583100
min        21.599787
10%        21.599787
25%        21.599787
50%        23.198294
75%        34.246469
90%        67.321552
95%        90.476101
max        99.999017
Name: raw_score, dtype: float64


In [35]:
counts = unflagged_covered['business_article_count']
for sf in [20, 30, 50, 80, 100]:
    scores = 20 + 80 * np.tanh(counts / sf)
    print(f"\nscale_factor = {sf}:")
    print(f"  median: {scores.median():.1f}, 75%: {scores.quantile(0.75):.1f}, "
          f"90%: {scores.quantile(0.9):.1f}, 95%: {scores.quantile(0.95):.1f}, "
          f"max: {scores.max():.1f}")


scale_factor = 20:
  median: 28.0, 75%: 53.8, 90%: 94.8, 95%: 99.8, max: 100.0

scale_factor = 30:
  median: 25.3, 75%: 43.3, 90%: 85.0, 95%: 98.4, max: 100.0

scale_factor = 50:
  median: 23.2, 75%: 34.2, 90%: 67.3, 95%: 90.5, max: 100.0

scale_factor = 80:
  median: 22.0, 75%: 29.0, 90%: 52.1, 95%: 75.8, max: 99.9

scale_factor = 100:
  median: 21.6, 75%: 27.2, 90%: 46.2, 95%: 67.8, max: 99.6


**Parameter selection**

Both parameters are chosen from the observed distribution of unflagged companies with business news.

- scale_factor = 30 controls how article count maps to score. A company needs about 30 articles to reach the middle of the scoring range, and about 60-80 to approach the cap; below 5 articles the score stays near the base of 20. We tested 20, 30, 50, 80, and 100 against the actual count distribution and chose 30 because it places the score's transition region across the 75th-90th percentile of the count distribution (9-34 articles). This gives clear score gradients across the middle and upper ranges without saturating too early.

- ambig_cap = 25 is set to the median score of unflagged companies at scale_factor = 30. Ambiguous matches cannot rank higher than a typical trustworthy company.

In [36]:
def compute_theme_score(row, scale_factor=50, ambig_cap=None):
    """
    Compute a 0-100 GKG-based score.
    
    - No GKG data or no business news: NaN
    - Business news + not ambiguous: 20 + 80 * tanh(count / scale_factor)
    - Business news + ambiguous: capped at ambig_cap (the median score of unflagged companies, 
      to keep suspicious companies below the "typical reliable company" level)

    """
    if not row['gkg_features_available']:
        return np.nan
    if not row['has_business_news']:
        return np.nan
    
    raw_score = 20 + 80 * np.tanh(row['business_article_count'] / scale_factor)
    
    if row['likely_ambiguous'] and ambig_cap is not None:
        return min(raw_score, ambig_cap)
    
    return raw_score

In [37]:
feature_table['theme_score'] = feature_table.apply(
    compute_theme_score,
    axis=1,
    scale_factor=30,
    ambig_cap=25
)

In [41]:
# Distribution of scores
print("Score distribution:")
print(feature_table['theme_score'].describe(
    percentiles=[.1, .25, .5, .75, .9, .95]
))

# How many are NaN vs valued
n_total = len(feature_table)
n_nan = feature_table['theme_score'].isna().sum()
n_scored = n_total - n_nan
print(f"\nNaN: {n_nan} ({n_nan/n_total:.2%})")
print(f"Scored: {n_scored} ({n_scored/n_total:.2%})")

# Break down by ambiguous flag
scored = feature_table[feature_table['theme_score'].notna()]
by_ambig = scored.groupby('likely_ambiguous')['theme_score'].describe()
print(f"\nScored companies by ambiguity flag:")
print(by_ambig)

# Top companies by score
top_scored = feature_table.sort_values('theme_score', ascending=False, na_position='last').head(20)
print(f"\nTop 20 by score:")
print(top_scored[['CompanyName', 'search_name', 'business_article_count', 
                   'likely_ambiguous', 'theme_score']].to_string())

Score distribution:
count    4223.000000
mean       34.665824
std        20.995357
min        22.665679
10%        22.665679
25%        22.665679
50%        25.000000
75%        33.211233
90%        68.349422
95%        95.985331
max       100.000000
Name: theme_score, dtype: float64

NaN: 95777 (95.78%)
Scored: 4223 (4.22%)

Scored companies by ambiguity flag:
                   count       mean        std        min        25%  \
likely_ambiguous                                                       
False             3097.0  38.340648  23.455407  22.665679  22.665679   
True              1126.0  24.558428   0.914619  22.665679  25.000000   

                        50%        75%    max  
likely_ambiguous                               
False             25.325446  43.305009  100.0  
True              25.000000  25.000000   25.0  

Top 20 by score:
                       CompanyName                 search_name  business_article_count  likely_ambiguous  theme_score
73141        GOLD R

In [42]:
feature_table.head()

,CompanyNumber,CompanyName,search_name,primary_sector,Accounts_AccountCategory,business_article_count,has_business_news,gkg_features_available,likely_ambiguous,theme_score
0,13209628,TICKETY BOO TEETH LIMITED,tickety boo teeth,Healthcare,UNAUDITED ABRIDGED,0,False,False,False,NaN
1,13887674,TECH INFUSION LIMITED,tech infusion,Fast growth & emerging sector,TOTAL EXEMPTION FULL,0,False,False,False,NaN
2,13408899,THE SCC ACADEMY LIMITED,the scc academy,Fast growth & emerging sector,TOTAL EXEMPTION FULL,0,False,False,False,NaN
3,SC374368,GRACEFRUIT LIMITED,gracefruit,Wholesale & Retail,TOTAL EXEMPTION FULL,0,False,False,False,NaN
4,13675979,LINHAM LIMITED,linham,Wholesale & Retail,TOTAL EXEMPTION FULL,0,False,False,True,NaN


In [43]:
# Save as both parquet (for analysis) and CSV (for portability)
feature_table.to_parquet(FEATURE_TABLE_THEME_SCORE_PQ, compression='snappy')
feature_table.to_csv(FEATURE_TABLE_THEME_SCORE_CSV, index=False)
print(f"Saved parquet: {FEATURE_TABLE_THEME_SCORE_PQ}")
print(f"Saved CSV: {FEATURE_TABLE_THEME_SCORE_CSV}")

Saved parquet: ..\output\gkg_features\gkg_features_theme_score_100k.parquet
Saved CSV: ..\output\gkg_features\gkg_features_theme_score_100k.csv


**Distribution observations**

- 4223 companies (4.22%) received a score. The remaining 95.78% are NaN, of which 94.29% have no GKG match and 1.49% have GKG matches without business themes.

- Flagged companies pin at or near the 25 cap (mean 24.6, max 25). The cap works.

- Unflagged companies are right-skewed: median 25, 75th percentile 43, max 100. The scoring range is used across all levels rather than compressed at one end.

**Limitations**

- The likely_ambiguous flag catches short single-token names and extreme article counts, but misses multi-token generic phrases (e.g. "gold reserves", "future energy") that structurally look like normal company names. Some false-match companies therefore reach the top of the ranking.

- More broadly, name-based matching against V2Organizations cannot fully resolve which specific organisation a mention refers to when multiple entities share the same name. Full disambiguation would require article-content analysis, which is out of scope.

- Given the low GKG coverage (~4% of companies), theme-based features are inherently a supplementary signal in the project, not a primary ranking source. The score should be interpreted as an indicator of media-signal strength for the covered subset, with known noise in the top range.